# SurfaceTarget-CRC — phase 01 — patch 1 18

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.18
#
# FULL-COHORT RNA-SEQ QUALITY CONTROL
#
# Objectives:
# 1. Validate processed expression matrix
# 2. Examine library sizes and gene detection
# 3. Calculate protein-coding detection at CPM thresholds
# 4. Analyze gene-assignment statistics
# 5. Stratify QC by project, cohort and workflow
# 6. Flag unusual samples using robust statistics
# 7. Generate publication-oriented QC figures
# 8. Save results and a processed-data checkpoint
#
# IMPORTANT:
# - No samples will be excluded.
# - No differential expression is performed.
# - CPM is used for descriptive QC only.
# - Raw counts remain unchanged.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import json


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

PROCESSED = ROOT / "data/processed"

RESULTS = ROOT / "results/tables"

FIGURES = ROOT / "results/figures"

CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    META,
    PROCESSED,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

GENES_PATH = (
    META / "tcga_gene_annotations.csv"
)

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

INITIAL_QC_PATH = (
    RESULTS / "tcga_expression_initial_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.18 — FULL-COHORT RNA-SEQ QC")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load processed expression matrix
# ------------------------------------------------------------

required_files = [
    MATRIX_PATH,
    GENES_PATH,
    SAMPLES_PATH,
    INITIAL_QC_PATH
]

for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    counts = archive["counts"]

    gene_ids = archive["gene_ids"]

    file_ids = archive["file_ids"]

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

initial_qc = pd.read_csv(
    INITIAL_QC_PATH,
    dtype={"file_id": str}
)

print("\nExpression matrix loaded successfully.")

print("Matrix dimensions:", counts.shape)

print("Gene annotations:", len(genes))

print("Sample records:", len(samples))


# ------------------------------------------------------------
# 3. Validate matrix consistency
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MATRIX INTEGRITY VALIDATION")
print("-" * 65)

if counts.shape != (60660, 472):

    raise RuntimeError(
        "Unexpected expression matrix dimensions."
    )

if not np.issubdtype(
    counts.dtype,
    np.integer
):

    raise RuntimeError(
        "Expression matrix contains non-integer values."
    )

if np.any(counts < 0):

    raise RuntimeError(
        "Negative raw counts detected."
    )

if not np.array_equal(
    gene_ids,
    genes["gene_id"].to_numpy()
):

    raise RuntimeError(
        "Gene identifier mismatch."
    )

if not np.array_equal(
    file_ids,
    samples["file_id"].to_numpy()
):

    raise RuntimeError(
        "Sample identifier mismatch."
    )

if not np.array_equal(
    file_ids,
    initial_qc["file_id"].to_numpy()
):

    raise RuntimeError(
        "Initial QC sample order mismatch."
    )

if samples["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate sample identifiers detected."
    )

required_sample_columns = [
    "file_id",
    "patient_id",
    "sample_id",
    "project",
    "dataset_group",
    "cohort_tier",
    "workflow_version"
]

missing_columns = [
    col for col in required_sample_columns
    if col not in samples.columns
]

if missing_columns:

    raise RuntimeError(
        f"Missing sample metadata: {missing_columns}"
    )

print("\nMatrix integrity: PASSED")

print("Gene annotation correspondence: PASSED")

print("Sample metadata correspondence: PASSED")


# ------------------------------------------------------------
# 4. Identify protein-coding genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE ANNOTATION SUMMARY")
print("-" * 65)

protein_coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

n_protein_coding = int(
    protein_coding_mask.sum()
)

print("\nTotal genes:", len(genes))

print(
    "Protein-coding genes:",
    n_protein_coding
)

if n_protein_coding != 19962:

    raise RuntimeError(
        "Unexpected protein-coding gene count."
    )


# ------------------------------------------------------------
# 5. Calculate expression QC metrics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING FULL-COHORT EXPRESSION QC")
print("-" * 65)

# Raw library sizes:
# Total unstranded counts assigned to genes.

library_sizes = counts.sum(
    axis=0,
    dtype=np.int64
)

if np.any(library_sizes <= 0):

    raise RuntimeError(
        "One or more samples have zero assigned counts."
    )

# Validate against previously saved measurements.

recorded_library_sizes = pd.to_numeric(
    initial_qc["assigned_gene_counts"],
    errors="raise"
).to_numpy(dtype=np.int64)

if not np.array_equal(
    library_sizes,
    recorded_library_sizes
):

    raise RuntimeError(
        "Library sizes differ from Patch 1.17."
    )

# All genes with at least one assigned count.

detected_all = np.count_nonzero(
    counts,
    axis=0
)

# Protein-coding counts.

coding_counts = counts[
    protein_coding_mask,
    :
]

detected_coding = np.count_nonzero(
    coding_counts,
    axis=0
)

# Descriptive CPM calculations.
#
# Denominator = all assigned unstranded gene counts.
#
# This does not replace downstream
# count-based statistical normalization.

cpm_scaling = (
    1_000_000.0
    / library_sizes.astype(np.float64)
)

coding_cpm = (
    coding_counts
    * cpm_scaling[np.newaxis, :]
)

coding_cpm1 = np.count_nonzero(
    coding_cpm >= 1,
    axis=0
)

coding_cpm5 = np.count_nonzero(
    coding_cpm >= 5,
    axis=0
)

coding_assigned_counts = coding_counts.sum(
    axis=0,
    dtype=np.int64
)

coding_count_fraction = (
    coding_assigned_counts
    / library_sizes
)

print("\nGene detection metrics calculated.")

print("CPM >= 1 detection: COMPLETE")

print("CPM >= 5 detection: COMPLETE")


# ------------------------------------------------------------
# 6. Prepare sample-level QC table
# ------------------------------------------------------------

qc = samples[
    required_sample_columns
].copy()

qc["assigned_gene_counts"] = (
    library_sizes
)

qc["library_size_millions"] = (
    library_sizes / 1e6
)

qc["detected_genes"] = (
    detected_all
)

qc["detected_protein_coding"] = (
    detected_coding
)

qc["protein_coding_cpm_ge1"] = (
    coding_cpm1
)

qc["protein_coding_cpm_ge5"] = (
    coding_cpm5
)

qc["protein_coding_count_fraction"] = (
    coding_count_fraction
)

# STAR summary counts from Patch 1.17.

star_columns = [
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
]

for column in star_columns:

    qc[column] = pd.to_numeric(
        initial_qc[column],
        errors="raise"
    ).to_numpy(dtype=np.int64)

star_total = (
    library_sizes
    + qc[star_columns].sum(
        axis=1
    ).to_numpy(dtype=np.int64)
)

if np.any(star_total <= 0):

    raise RuntimeError(
        "Invalid STAR counting totals."
    )

qc["assignment_percent"] = (
    100 * library_sizes / star_total
)

qc["no_feature_percent"] = (
    100 * qc["N_noFeature"]
    / star_total
)

qc["ambiguous_percent"] = (
    100 * qc["N_ambiguous"]
    / star_total
)

qc["multimapping_percent"] = (
    100 * qc["N_multimapping"]
    / star_total
)

qc["unmapped_percent"] = (
    100 * qc["N_unmapped"]
    / star_total
)

# Workflow identifiers are categorical.
# A hexadecimal workflow ID does not,
# by itself, establish chronological order.

qc["workflow_short"] = (
    qc["workflow_version"].str[:12]
)

qc["analysis_group"] = (
    qc["project"]
    + " | "
    + np.where(
        qc["dataset_group"].eq("Normal"),
        "Normal",
        qc["cohort_tier"]
    )
)

print("\nSample-level QC table constructed.")

print("Samples:", len(qc))


# ------------------------------------------------------------
# 7. Validate cohort composition
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COHORT COMPOSITION VALIDATION")
print("-" * 65)

expected_groups = {

    "TCGA-COAD | Tier A": 98,

    "TCGA-COAD | Tier B": 194,

    "TCGA-COAD | Normal": 41,

    "TCGA-READ | Tier A": 46,

    "TCGA-READ | Tier B": 83,

    "TCGA-READ | Normal": 10
}

group_order = list(
    expected_groups.keys()
)

observed_groups = (
    qc["analysis_group"]
    .value_counts()
    .to_dict()
)

if observed_groups != expected_groups:

    raise RuntimeError(
        "Cohort composition differs from expected values."
    )

print("\nAll six analysis groups validated.")

print(
    qc["analysis_group"]
    .value_counts()
    .reindex(group_order)
    .to_string()
)


# ------------------------------------------------------------
# 8. Full-cohort QC summary
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FULL-COHORT QUALITY SUMMARY")
print("-" * 65)

qc_summary = (
    qc.groupby("analysis_group")
    .agg(

        samples=(
            "file_id",
            "count"
        ),

        median_library_millions=(
            "library_size_millions",
            "median"
        ),

        median_detected_genes=(
            "detected_genes",
            "median"
        ),

        median_protein_coding_cpm1=(
            "protein_coding_cpm_ge1",
            "median"
        ),

        median_assignment_percent=(
            "assignment_percent",
            "median"
        ),

        median_no_feature_percent=(
            "no_feature_percent",
            "median"
        )

    )
    .reindex(group_order)
    .reset_index()
)

print("\n")
print(
    qc_summary
    .round(2)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. Library-size distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LIBRARY-SIZE DISTRIBUTION")
print("-" * 65)

library_summary = (
    qc.groupby("analysis_group")[
        "library_size_millions"
    ]
    .agg(
        [
            "min",
            "median",
            "max"
        ]
    )
    .reindex(group_order)
)

print("\n")
print(
    library_summary
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# 10. Workflow-stratified quality assessment
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("QUALITY METRICS BY WORKFLOW VERSION")
print("-" * 65)

workflow_qc = (
    qc.groupby(
        [
            "analysis_group",
            "workflow_short"
        ]
    )
    .agg(

        samples=(
            "file_id",
            "count"
        ),

        median_library_millions=(
            "library_size_millions",
            "median"
        ),

        median_coding_cpm1=(
            "protein_coding_cpm_ge1",
            "median"
        ),

        median_assignment_percent=(
            "assignment_percent",
            "median"
        ),

        median_no_feature_percent=(
            "no_feature_percent",
            "median"
        )

    )
    .reset_index()
)

workflow_qc["analysis_group"] = (
    pd.Categorical(
        workflow_qc["analysis_group"],
        categories=group_order,
        ordered=True
    )
)

workflow_qc = (
    workflow_qc
    .sort_values(
        [
            "analysis_group",
            "workflow_short"
        ]
    )
)

print("\n")
print(
    workflow_qc
    .round(2)
    .to_string(index=False)
)

print("\nIMPORTANT:")
print(
    "Workflow and cohort associations are "
    "descriptive. These tables do not "
    "establish a causal technical effect."
)


# ------------------------------------------------------------
# 11. Robust sample outlier screening
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ROBUST SAMPLE OUTLIER SCREENING")
print("-" * 65)

# Compare samples only within their own
# project and cohort/tissue group.
#
# Robust z-score = 0.67448975 * (x - median) / MAD
#
# |z| > 3.5 is used here only as a
# provisional review threshold.
#
# No samples will be removed.

qc["log10_library_size"] = np.log10(
    qc["assigned_gene_counts"]
)

metrics_to_screen = {

    "log10_library_size": "low",

    "protein_coding_cpm_ge1": "low",

    "assignment_percent": "low",

    "no_feature_percent": "high"
}

flag_reasons = {
    index: []
    for index in qc.index
}

flag_records = []

for group_name, group in qc.groupby(
    "analysis_group"
):

    group_indices = group.index.to_numpy()

    if len(group) < 10:
        continue

    for metric, direction in metrics_to_screen.items():

        values = group[
            metric
        ].to_numpy(dtype=float)

        median = np.median(values)

        mad = np.median(
            np.abs(values - median)
        )

        if not np.isfinite(mad) or mad == 0:

            # Cannot calculate a meaningful
            # robust score with zero MAD.

            continue

        robust_z = (
            0.67448975
            * (values - median)
            / mad
        )

        if direction == "low":

            flagged = robust_z < -3.5

        else:

            flagged = robust_z > 3.5

        for local_position in np.flatnonzero(flagged):

            row_index = group_indices[
                local_position
            ]

            reason = (
                f"{metric}: "
                f"robust_z={robust_z[local_position]:.2f}"
            )

            flag_reasons[
                row_index
            ].append(reason)

            flag_records.append({

                "file_id":
                    qc.loc[row_index, "file_id"],

                "patient_id":
                    qc.loc[row_index, "patient_id"],

                "analysis_group":
                    group_name,

                "metric":
                    metric,

                "observed_value":
                    float(values[local_position]),

                "group_median":
                    float(median),

                "robust_z":
                    float(
                        robust_z[local_position]
                    )
            })

qc["qc_review_flags"] = [
    "; ".join(
        flag_reasons[index]
    )
    for index in qc.index
]

qc["n_qc_review_flags"] = [
    len(
        flag_reasons[index]
    )
    for index in qc.index
]

flag_details = pd.DataFrame(
    flag_records,
    columns=[
        "file_id",
        "patient_id",
        "analysis_group",
        "metric",
        "observed_value",
        "group_median",
        "robust_z"
    ]
)

review_samples = qc[
    qc["n_qc_review_flags"] > 0
].copy()

review_samples = (
    review_samples
    .sort_values(
        [
            "n_qc_review_flags",
            "analysis_group"
        ],
        ascending=[
            False,
            True
        ]
    )
)

print(
    "\nSamples flagged for review:",
    len(review_samples)
)

if not review_samples.empty:

    print("\nFlagged samples (first 20):")

    print(
        review_samples[
            [
                "patient_id",
                "sample_id",
                "analysis_group",
                "workflow_short",
                "library_size_millions",
                "protein_coding_cpm_ge1",
                "assignment_percent",
                "no_feature_percent",
                "n_qc_review_flags"
            ]
        ]
        .head(20)
        .round(2)
        .to_string(index=False)
    )

else:

    print(
        "\nNo samples crossed the provisional "
        "outlier-review thresholds."
    )

print("\nNo samples have been excluded.")


# ------------------------------------------------------------
# 12. Generate QC figures
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING RNA-SEQ QC FIGURES")
print("-" * 65)

figure_specs = [

    (
        "library_size_millions",
        "Assigned gene counts (millions)",
        "tcga_qc_library_sizes.png",
        "RNA-seq library size by cohort"
    ),

    (
        "protein_coding_cpm_ge1",
        "Protein-coding genes with CPM ≥ 1",
        "tcga_qc_protein_coding_detection.png",
        "Protein-coding gene detection by cohort"
    ),

    (
        "assignment_percent",
        "Gene-assignment fraction (%)",
        "tcga_qc_assignment_fraction.png",
        "STAR gene assignment by cohort"
    )

]

for metric, ylabel, filename, title in figure_specs:

    plot_data = [

        qc.loc[
            qc["analysis_group"] == group,
            metric
        ].to_numpy(dtype=float)

        for group in group_order
    ]

    fig, ax = plt.subplots(
        figsize=(11, 5.5)
    )

    ax.boxplot(
        plot_data,
        showfliers=True
    )

    ax.set_xticks(
        np.arange(
            1,
            len(group_order) + 1
        )
    )

    ax.set_xticklabels(
        group_order,
        rotation=35,
        ha="right"
    )

    ax.set_ylabel(ylabel)

    ax.set_title(title)

    ax.grid(
        axis="y",
        alpha=0.25
    )

    fig.tight_layout()

    figure_path = FIGURES / filename

    fig.savefig(
        figure_path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)

    print(
        "Saved:",
        filename
    )


# ------------------------------------------------------------
# 13. Save QC tables
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FULL-COHORT QC RESULTS")
print("-" * 65)

qc_path = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

summary_path = (
    RESULTS / "tcga_full_cohort_qc_summary.csv"
)

workflow_path = (
    RESULTS / "tcga_full_cohort_qc_by_workflow.csv"
)

review_path = (
    RESULTS / "tcga_full_cohort_qc_review_samples.csv"
)

flag_path = (
    RESULTS / "tcga_full_cohort_qc_flag_details.csv"
)

qc.to_csv(
    qc_path,
    index=False
)

qc_summary.to_csv(
    summary_path,
    index=False
)

workflow_qc.to_csv(
    workflow_path,
    index=False
)

review_samples.to_csv(
    review_path,
    index=False
)

flag_details.to_csv(
    flag_path,
    index=False
)

print("\nAll QC tables saved.")


# ------------------------------------------------------------
# 14. Save research provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.18",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "input_matrix": str(MATRIX_PATH),

    "matrix_dimensions": [
        int(counts.shape[0]),
        int(counts.shape[1])
    ],

    "samples_assessed": int(len(qc)),

    "protein_coding_genes": int(
        n_protein_coding
    ),

    "count_column": "unstranded",

    "cpm_denominator": (
        "Total unstranded counts assigned "
        "to all annotated genes"
    ),

    "qc_metrics": [
        "Assigned gene counts",
        "Detected genes",
        "Detected protein-coding genes",
        "Protein-coding detection at CPM >= 1",
        "Protein-coding detection at CPM >= 5",
        "STAR gene-assignment fraction",
        "STAR no-feature fraction",
        "STAR ambiguous fraction"
    ],

    "outlier_screening": {

        "method": (
            "Median absolute deviation "
            "robust z-score"
        ),

        "threshold": 3.5,

        "reference_groups": (
            "Project by tissue/cohort tier"
        ),

        "flags_are_exclusions": False

    },

    "flagged_samples": int(
        len(review_samples)
    ),

    "samples_excluded": 0,

    "normalization_performed": False,

    "descriptive_cpm_calculated": True,

    "differential_expression_performed": False,

    "limitations": [

        "QC flags require individual review.",

        "No sequencing-level FastQC or "
        "alignment QC was independently performed.",

        "Gene-assignment fraction is not "
        "genome alignment rate.",

        "CPM detection does not establish "
        "biological sample quality.",

        "Workflow version and cohort tier "
        "are strongly associated.",

        "Normal tissue comparisons require "
        "additional statistical design.",

        "Tier B is computationally low-MSI "
        "rather than categorically verified MSS."

    ]
}

provenance_path = (
    META / "tcga_full_cohort_qc_provenance.json"
)

with open(
    provenance_path,
    "w"
) as output:

    json.dump(
        provenance,
        output,
        indent=4
    )


# ------------------------------------------------------------
# 15. Create processed-data checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PROCESSED-DATA CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint_path = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_PostMatrix_Patch118_{timestamp}.zip"
)

# This checkpoint includes:
#
# - Processed count matrix
# - Sample and gene metadata
# - QC tables
# - QC figures
# - Existing project notes/code
#
# It intentionally excludes the 1.86 GiB
# original RNA-seq TSV files.

checkpoint_folders = [
    "data/metadata",
    "data/processed",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint_path,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in checkpoint_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )

checkpoint_size_mib = (
    checkpoint_path.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint successfully created.")

print(
    f"Checkpoint size: "
    f"{checkpoint_size_mib:.2f} MiB"
)

print("\nCheckpoint path:")

print(checkpoint_path)


# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.18 SUMMARY")
print("=" * 65)

print("\nSamples analyzed:", len(qc))

print(
    "Protein-coding genes analyzed:",
    n_protein_coding
)

print(
    "\nSamples flagged for review:",
    len(review_samples)
)

print("Samples excluded: 0")

print("\nQC tables saved: 5")

print("QC figures saved: 3")

print("\nProcessed-data checkpoint: CREATED")

print(
    f"Checkpoint size: "
    f"{checkpoint_size_mib:.2f} MiB"
)

print("\nNo additional data downloaded.")

print("No changes made to raw expression counts.")

print("No differential expression performed.")

print("\nPATCH 1.18 COMPLETE")

print("=" * 65)